# 06. 지속성과 메모리 — 체크포인터와 메모리 스토어

## 학습 목표

체크포인터로 상태를 저장하고, 스토어로 장기 메모리를 구현합니다.

- **체크포인터**: 각 실행 단계의 상태를 자동으로 저장하고 복원
- **상태 조회**: `get_state()`와 `get_state_history()`로 저장된 상태 확인
- **상태 수정**: `update_state()`로 외부에서 상태 변경
- **스레드 독립성**: 서로 다른 `thread_id`는 완전히 독립된 상태
- **InMemoryStore**: 스레드 간 공유되는 장기 메모리 (standalone 및 그래프 연동)
- **대화 길이 관리**: `trim_messages`와 `RemoveMessage`로 메시지 관리
- **Durable Execution**: 실패 시 마지막 체크포인트에서 재개

## 6.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")

In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 6.2 체크포인터 — 각 실행 단계의 상태를 자동으로 저장합니다

LangGraph는 세 가지 체크포인터를 제공합니다:

- **`InMemorySaver`**: 개발용 (메모리에 저장, 프로세스 종료 시 삭제)
- **`SqliteSaver`**: 로컬 개발 (파일에 저장, 재시작 후에도 유지)
- **`PostgresSaver`**: 프로덕션 (DB에 저장, 확장 가능)

체크포인터를 `compile()`에 전달하면, 그래프의 각 노드 실행 후 자동으로 상태가 저장됩니다.

In [3]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

def chatbot(state: MessagesState) -> dict:
    response = model.invoke(state["messages"], config=lf_config)
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# 체크포인터와 함께 컴파일
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "session-1"}}

# 대화 1
r1 = graph.invoke({"messages": [HumanMessage(content="제 이름은 앨리스입니다.")]}, {**config, **lf_config})
print("턴 1:", r1["messages"][-1].content)

# 대화 2 — 이전 대화를 기억
r2 = graph.invoke({"messages": [HumanMessage(content="제 이름이 뭐였죠?")]}, {**config, **lf_config})
print("턴 2:", r2["messages"][-1].content)

턴 1: 안녕하세요, 앨리스님! 만나서 반가워요. 오늘 어떻게 도와드릴까요?


턴 2: 앨리스님이라고 하셨죠! 도움이 필요하시면 언제든 말씀해 주세요. 😊


## 6.3 get_state() — 현재 저장된 상태 조회

`get_state()`는 지정된 스레드의 최신 체크포인트 상태를 반환합니다.
메시지 수, 체크포인트 ID, 다음 실행할 노드 등의 정보를 확인할 수 있습니다.

In [4]:
state = graph.get_state(config)
print(f"스레드: {config['configurable']['thread_id']}")
print(f"메시지 수: {len(state.values['messages'])}")
print(f"체크포인트 ID: {state.config['configurable']['checkpoint_id']}")
print(f"다음 노드: {state.next}")

스레드: session-1
메시지 수: 4
체크포인트 ID: 1f1196e1-c938-68ee-8004-4f52838ad157
다음 노드: ()


## 6.4 get_state_history() — 전체 실행 이력 조회

`get_state_history()`는 해당 스레드의 모든 체크포인트를 최신순으로 반환합니다.
이를 통해 그래프 실행의 전체 이력을 추적할 수 있습니다.

In [5]:
print("상태 이력 (최신순):")
for i, snapshot in enumerate(graph.get_state_history(config)):
    msg_count = len(snapshot.values.get("messages", []))
    print(f"  [{i}] 체크포인트={snapshot.config['configurable']['checkpoint_id'][:20]}... 메시지={msg_count}")
    if i >= 4:
        print("  ... (생략)")
        break

상태 이력 (최신순):
  [0] 체크포인트=1f1196e1-c938-68ee-8... 메시지=4
  [1] 체크포인트=1f1196e1-bf0f-6ddf-8... 메시지=3
  [2] 체크포인트=1f1196e1-bf0f-6dde-8... 메시지=2
  [3] 체크포인트=1f1196e1-bf0a-6139-8... 메시지=2
  [4] 체크포인트=1f1196e1-abd9-65c9-8... 메시지=1
  ... (생략)


## 6.5 update_state() — 저장된 상태를 외부에서 수정

`update_state()`를 사용하면 체크포인트에 저장된 상태를 프로그래밍 방식으로 수정할 수 있습니다.
예를 들어, 시스템 노트를 추가하거나 사용자 선호도를 반영하는 등의 작업이 가능합니다.

In [6]:
from langchain.messages import AIMessage

# 상태에 메시지 직접 추가
graph.update_state(config, {
    "messages": [AIMessage(content="(시스템 노트: 사용자는 한국어 응답을 선호합니다.)")]
})

# 수정된 상태로 계속 대화
r3 = graph.invoke({"messages": [HumanMessage(content="재미있는 사실을 알려주세요.")]}, {**config, **lf_config})
print("업데이트 후:", r3["messages"][-1].content[:200])

업데이트 후: 물론이죠, 앨리스님! 재미있는 사실 하나 알려드릴게요.

**문어는 세 개의 심장을 가지고 있습니다.**  
문어는 두 개의 심장이 아가미로 혈액을 보내고, 나머지 한 개의 심장은 몸 전체로 산소가 풍부한 혈액을 보낸답니다. 게다가 문어의 혈액은 산소를 운반하기 위해 '헤모시아닌'이라는 구리 기반의 분자를 사용하는데, 그래서 혈액 색이 파란색이에요!

신기


## 6.6 스레드 독립성 — 다른 thread_id는 완전히 독립된 상태

각 `thread_id`는 완전히 독립된 대화 상태를 가집니다.
다른 스레드의 대화 내용은 서로 영향을 주지 않습니다.

In [7]:
config_b = {"configurable": {"thread_id": "session-2"}}
r = graph.invoke({"messages": [HumanMessage(content="제 이름을 아시나요?")]}, {**config_b, **lf_config})
print("다른 스레드:", r["messages"][-1].content)
print("-> 다른 thread_id이므로 이전 대화를 알 수 없음")

다른 스레드: 아쉽게도 저는 당신의 이름을 알지 못합니다. 대화가 익명으로 이루어지기 때문에 사용자의 이름이나 개인정보는 알 수 없습니다. 원하신다면 이름을 알려주셔도 좋고, 닉네임을 사용하셔도 괜찮아요!
-> 다른 thread_id이므로 이전 대화를 알 수 없음


## 6.7 InMemoryStore — 스레드 간 공유 장기 메모리

`InMemoryStore`는 스레드 간에 공유되는 키-값 저장소입니다.
사용자 프로필, 선호도 등 스레드를 넘어 유지해야 하는 정보를 저장하는 데 사용합니다.

- `put()`: 네임스페이스와 키로 데이터 저장
- `get()`: 특정 항목 조회
- `search()`: 네임스페이스 내 검색

In [8]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# 데이터 저장
store.put(("users",), "alice", {"favorite_color": "blue", "city": "Seoul"})
store.put(("users",), "bob", {"favorite_color": "red", "city": "Tokyo"})

# 데이터 조회
alice = store.get(("users",), "alice")
print(f"Alice: {alice.value}")

# 검색
results = store.search(("users",))
print(f"\n전체 사용자 ({len(results)}명):")
for item in results:
    print(f"  {item.key}: {item.value}")

Alice: {'favorite_color': 'blue', 'city': 'Seoul'}

전체 사용자 (2명):
  alice: {'favorite_color': 'blue', 'city': 'Seoul'}
  bob: {'favorite_color': 'red', 'city': 'Tokyo'}


### 6.7.5 InMemoryStore를 그래프와 함께 사용하기

`InMemoryStore`를 `compile(store=store)`로 그래프에 전달하면, 각 노드 함수에서 `store` 파라미터를 통해 스토어에 직접 접근할 수 있습니다.
이 패턴을 사용하면 노드 안에서 사용자 정보를 저장하고 조회하여, 스레드를 넘어 장기 메모리를 유지할 수 있습니다.

- `compile(store=store)`: 그래프에 스토어 연결
- 노드 함수에 `store` 파라미터 추가: LangGraph가 자동으로 스토어 인스턴스를 주입
- `config["configurable"]["user_id"]`로 사용자별 네임스페이스 분리

In [9]:
import uuid
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langchain_core.runnables import RunnableConfig
from langchain.messages import HumanMessage

# --- Store and checkpointer ---
memory_store = InMemoryStore()
memory_checkpointer = InMemorySaver()

# --- Node that reads/writes long-term memory via the store parameter ---
def chatbot_with_memory(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    """store 파라미터를 통해 장기 메모리를 읽고 쓰는 노드입니다."""
    user_id = config["configurable"].get("user_id", "anonymous")
    namespace = ("memories", user_id)

    # 이 사용자의 기존 메모리를 조회
    existing = store.search(namespace)
    memory_text = "\n".join(f"- {item.value['data']}" for item in existing)

    # 메모리에서 시스템 컨텍스트 구성
    system_msg = "당신은 유용한 어시스턴트입니다."
    if memory_text:
        system_msg += f"\n\n사용자 메모리:\n{memory_text}"

    messages = [{"role": "system", "content": system_msg}] + state["messages"]
    response = model.invoke(messages, config=lf_config)

    # 사용자가 자신에 대해 언급한 새로운 사실을 저장
    user_msg = state["messages"][-1].content.lower()
    if any(kw in user_msg for kw in ["좋아", "사랑", "이름은", "선호", "즐겨"]):
        store.put(namespace, str(uuid.uuid4()), {"data": state["messages"][-1].content})

    return {"messages": [response]}

# --- Build graph ---
builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot_with_memory)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph_with_store = builder.compile(
    checkpointer=memory_checkpointer,
    store=memory_store,  # <-- store를 그래프에 연결
)

# --- Thread 1: user shares preferences ---
config_1 = {"configurable": {"thread_id": "store-thread-1", "user_id": "user-alice"}}
r1 = graph_with_store.invoke(
    {"messages": [HumanMessage(content="저는 등산과 사진 찍기를 좋아합니다.")]},
    {**config_1, **lf_config},
)
print("스레드 1:", r1["messages"][-1].content[:150])

# --- Thread 2 (different thread, same user): memories persist ---
config_2 = {"configurable": {"thread_id": "store-thread-2", "user_id": "user-alice"}}
r2 = graph_with_store.invoke(
    {"messages": [HumanMessage(content="제 취미가 뭔가요?")]},
    {**config_2, **lf_config},
)
print("\n스레드 2 (새 스레드, 같은 사용자):", r2["messages"][-1].content[:150])

# --- Verify stored memories ---
stored = memory_store.search(("memories", "user-alice"))
print(f"\nuser-alice의 저장된 메모리 ({len(stored)}개):")
for item in stored:
    print(f"  {item.key[:8]}...: {item.value}")

스레드 1: 정말 멋진 취미를 가지고 계시네요! 등산과 사진은 자연의 아름다움을 더욱 깊이 느낄 수 있는 최고의 조합인 것 같아요. 주로 어떤 산이나 지역을 자주 가시나요? 그리고 어떤 풍경이나 피사체를 찍는 것을 좋아하시나요? 필요하다면 등산 코스 추천이나 자연 사진 촬영 팁도 



스레드 2 (새 스레드, 같은 사용자): 당신의 취미는 등산과 사진 찍기입니다.

user-alice의 저장된 메모리 (1개):
  fa936e5e...: {'data': '저는 등산과 사진 찍기를 좋아합니다.'}


## 6.7.6 대화 길이 관리 — trim_messages와 RemoveMessage

대화가 길어지면 LLM의 컨텍스트 윈도우를 초과할 수 있습니다. LangGraph는 두 가지 방법으로 메시지를 관리합니다:

### `trim_messages`
- 토큰 수 기준으로 오래된 메시지를 자동으로 잘라냅니다
- `strategy="last"`: 최근 메시지만 유지
- `start_on="human"`: 잘린 결과가 항상 사용자 메시지로 시작하도록 보장
- 원본 상태는 수정하지 않고, 잘린 메시지 목록을 반환합니다 (체크포인트에는 전체 이력이 유지됨)

### `RemoveMessage`
- 특정 메시지를 체크포인트에서 영구적으로 삭제합니다
- `MessagesState`의 리듀서가 `RemoveMessage`를 감지하여 해당 메시지를 제거합니다
- 오래된 메시지를 정리하여 저장 공간을 절약할 때 유용합니다

In [10]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages.utils import trim_messages, count_tokens_approximately
from langchain.messages import HumanMessage, AIMessage, RemoveMessage

# ============================================================
# Part 1: trim_messages — LLM 호출 전에 메시지 잘라내기
# ============================================================

def chatbot_with_trim(state: MessagesState) -> dict:
    """토큰 예산에 맞게 메시지를 잘라낸 후 LLM을 호출합니다."""
    trimmed = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens=80,
        start_on="human",
        end_on=("human", "tool"),
    )
    print(f"  원본 메시지: {len(state['messages'])}개, 잘라낸 후: {len(trimmed)}개")
    response = model.invoke(trimmed, config=lf_config)
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot_with_trim)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

trim_checkpointer = InMemorySaver()
trim_graph = builder.compile(checkpointer=trim_checkpointer)
trim_config = {"configurable": {"thread_id": "trim-demo"}}

# Build up a long conversation
for i, msg in enumerate(["안녕하세요, 제 이름은 앨리스입니다.", "저는 서울에 살고 있습니다.", "저는 엔지니어로 일하고 있습니다.", "제 소개 내용을 정리해 주세요."]):
    print(f"턴 {i+1}: {msg}")
    r = trim_graph.invoke({"messages": [HumanMessage(content=msg)]}, {**trim_config, **lf_config})
    print(f"  AI: {r['messages'][-1].content[:120]}\n")

# Verify full history is preserved in checkpoint despite trimming
full_state = trim_graph.get_state(trim_config)
print(f"체크포인트의 총 메시지 수: {len(full_state.values['messages'])}")
print("-> trim_messages는 LLM 호출 시에만 잘라내고, 체크포인트에는 전체 이력이 보존됩니다")

턴 1: 안녕하세요, 제 이름은 앨리스입니다.
  원본 메시지: 1개, 잘라낸 후: 1개


  AI: 안녕하세요, 앨리스님! 만나서 반가워요. 오늘은 어떤 도움을 드릴 수 있을까요? 😊

턴 2: 저는 서울에 살고 있습니다.
  원본 메시지: 3개, 잘라낸 후: 3개


  AI: 서울에 사시는군요! 서울은 정말 멋진 도시죠. 혹시 서울에 관련된 정보나 추천을 원하시나요, 아니면 다른 주제로 이야기하고 싶으신가요? 어떤 내용이든 편하게 말씀해 주세요! 😊

턴 3: 저는 엔지니어로 일하고 있습니다.
  원본 메시지: 5개, 잘라낸 후: 5개


  AI: 와, 엔지니어로 일하시는군요! 멋지세요. 혹시 어떤 분야의 엔지니어인지 여쭤봐도 될까요? 소프트웨어, 하드웨어, 토목, 기계 등 여러 분야가 있을 것 같아서요. 엔지니어로서 일하면서 궁금한 점이나 도움이 필요하신 것

턴 4: 제 소개 내용을 정리해 주세요.
  원본 메시지: 7개, 잘라낸 후: 3개


  AI: 물론입니다! 아래와 같이 간단하게 소개 내용을 정리해 드릴 수 있습니다.

---

**소개 예시**

저는 엔지니어로 일하고 있습니다.

---

더 구체적으로 작성하고 싶다면, 예를 들어 아래와 같이 추가하실 수

체크포인트의 총 메시지 수: 8
-> trim_messages는 LLM 호출 시에만 잘라내고, 체크포인트에는 전체 이력이 보존됩니다


In [11]:
# ============================================================
# Part 2: RemoveMessage — 체크포인트에서 메시지 영구 삭제
# ============================================================
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, RemoveMessage

def chatbot_node(state: MessagesState) -> dict:
    response = model.invoke(state["messages"], config=lf_config)
    return {"messages": [response]}

def delete_old_messages(state: MessagesState) -> dict:
    """최근 2개 메시지만 유지하고 나머지를 체크포인트에서 삭제합니다."""
    messages = state["messages"]
    if len(messages) > 2:
        to_delete = messages[:-2]
        print(f"  체크포인트에서 {len(to_delete)}개의 오래된 메시지 삭제 중")
        return {"messages": [RemoveMessage(id=m.id) for m in to_delete]}
    return {"messages": []}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot_node)
builder.add_node("cleanup", delete_old_messages)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", "cleanup")
builder.add_edge("cleanup", END)

rm_checkpointer = InMemorySaver()
rm_graph = builder.compile(checkpointer=rm_checkpointer)
rm_config = {"configurable": {"thread_id": "remove-demo"}}

# Run several turns
for msg_text in ["안녕하세요! 저는 Bob입니다.", "저는 요리를 좋아합니다.", "제 이름이 뭐였죠?"]:
    r = rm_graph.invoke({"messages": [HumanMessage(content=msg_text)]}, {**rm_config, **lf_config})
    state = rm_graph.get_state(rm_config)
    print(f"사용자: {msg_text}")
    print(f"  AI: {r['messages'][-1].content[:120]}")
    print(f"  체크포인트 내 메시지 수: {len(state.values['messages'])}\n")

print("-> RemoveMessage로 오래된 메시지를 체크포인트에서 영구적으로 삭제하여 저장 공간을 절약합니다")

사용자: 안녕하세요! 저는 Bob입니다.
  AI: 안녕하세요, Bob님! 만나서 반가워요. 무엇을 도와드릴까요? 😊
  체크포인트 내 메시지 수: 2



  체크포인트에서 2개의 오래된 메시지 삭제 중
사용자: 저는 요리를 좋아합니다.
  AI: 정말 멋지네요, Bob님! 어떤 요리를 자주 하시나요? 아니면 좋아하는 요리나 배우고 싶은 요리가 있으신가요? 요리 레시피, 팁, 혹은 추천 메뉴 등 어떤 것이든 도와드릴 수 있어요!
  체크포인트 내 메시지 수: 2



  체크포인트에서 2개의 오래된 메시지 삭제 중
사용자: 제 이름이 뭐였죠?
  AI: 아, 죄송합니다! 이전 답변에서 실수로 "Bob님"이라고 불렀어요. 실제로는 사용자님께서 이름을 알려주신 적이 없어요. 혹시 원하시는 호칭이나 이름이 있으신가요? 알려주시면 앞으로 그렇게 불러드릴게요!
  체크포인트 내 메시지 수: 2

-> RemoveMessage로 오래된 메시지를 체크포인트에서 영구적으로 삭제하여 저장 공간을 절약합니다


## 6.8 Durable Execution — 실패 시 마지막 체크포인트에서 재개

체크포인터를 사용하면 **Durable Execution**이 가능합니다.
그래프 실행 중 오류가 발생하더라도, 마지막으로 성공한 체크포인트에서 재개할 수 있습니다.
이미 완료된 노드는 다시 실행하지 않으므로 비용과 시간을 절약합니다.

아래 예제에서는 3단계 파이프라인을 구성합니다:
1. **step_1**: 데이터 수집 (항상 성공)
2. **step_2**: 데이터 분석 (항상 성공)
3. **step_3**: 외부 API 호출 (첫 번째 실행에서 실패, 재시도 시 성공)

`attempt_count`를 통해 첫 실행에서 step_3이 실패하고, 두 번째 `invoke()` 호출 시 step_1과 step_2를 건너뛰고 step_3에서만 재개되는 것을 확인합니다.

In [12]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# --- State ---
class PipelineState(TypedDict):
    data: str
    log: Annotated[list[str], operator.add]

# Track how many times step_3 has been called across invocations
attempt_count = {"step_3": 0}

# --- Nodes ---
def step_1(state: PipelineState) -> dict:
    """Step 1: 데이터 수집 — 항상 성공합니다."""
    print("  >> step_1 실행됨 (데이터 수집)")
    return {
        "data": "수집된 원시 데이터",
        "log": ["step_1: 데이터 수집 완료"],
    }

def step_2(state: PipelineState) -> dict:
    """Step 2: 데이터 분석 — 항상 성공합니다."""
    print("  >> step_2 실행됨 (분석)")
    return {
        "data": state["data"] + " -> 분석 완료",
        "log": ["step_2: 데이터 분석 완료"],
    }

def step_3(state: PipelineState) -> dict:
    """Step 3: 외부 API 호출 — 첫 번째 시도에서 실패하고, 재시도 시 성공합니다."""
    attempt_count["step_3"] += 1
    print(f"  >> step_3 실행됨 (API 호출, 시도 {attempt_count['step_3']})")
    if attempt_count["step_3"] == 1:
        raise ConnectionError("시뮬레이션된 API 오류!")
    return {
        "data": state["data"] + " -> API 결과 수신",
        "log": ["step_3: API 호출 성공"],
    }

# --- Build graph ---
builder = StateGraph(PipelineState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

durable_checkpointer = InMemorySaver()
durable_graph = builder.compile(checkpointer=durable_checkpointer)

config = {"configurable": {"thread_id": "durable-1"}}

# --- First invocation: step_3 will fail ---
print("=== 1차 호출: step_3에서 실패 예상 ===")
try:
    durable_graph.invoke({"data": "", "log": []}, {**config, **lf_config})
except ConnectionError as e:
    print(f"  !! 오류 발생: {e}")

# Check checkpoint state after failure
state_after_fail = durable_graph.get_state(config)
print(f"\n실패 후 체크포인트:")
print(f"  다음 실행할 노드: {state_after_fail.next}")
print(f"  현재까지 로그: {state_after_fail.values.get('log', [])}")

# --- Second invocation: resumes from step_3 only ---
print("\n=== 2차 호출: 마지막 체크포인트에서 재개 ===")
result = durable_graph.invoke(None, {**config, **lf_config})
print(f"\n최종 결과:")
print(f"  데이터: {result['data']}")
print(f"  로그: {result['log']}")
print("\n-> step_1, step_2는 재실행되지 않고 step_3만 재실행되었습니다 (durable execution)")

=== 1차 호출: step_3에서 실패 예상 ===
  >> step_1 실행됨 (데이터 수집)
  >> step_2 실행됨 (분석)
  >> step_3 실행됨 (API 호출, 시도 1)
  !! 오류 발생: 시뮬레이션된 API 오류!

실패 후 체크포인트:
  다음 실행할 노드: ('step_3',)
  현재까지 로그: ['step_1: 데이터 수집 완료', 'step_2: 데이터 분석 완료']

=== 2차 호출: 마지막 체크포인트에서 재개 ===
  >> step_3 실행됨 (API 호출, 시도 2)

최종 결과:
  데이터: 수집된 원시 데이터 -> 분석 완료 -> API 결과 수신
  로그: ['step_1: 데이터 수집 완료', 'step_2: 데이터 분석 완료', 'step_3: API 호출 성공']

-> step_1, step_2는 재실행되지 않고 step_3만 재실행되었습니다 (durable execution)


## 6.9 요약

| 개념 | 설명 |
|------|------|
| **체크포인터** | 각 노드 실행 후 상태를 자동 저장 (`InMemorySaver`, `SqliteSaver`, `PostgresSaver`) |
| `get_state()` | 현재 스레드의 최신 체크포인트 상태 조회 |
| `get_state_history()` | 스레드의 전체 체크포인트 이력 조회 (최신순) |
| `update_state()` | 저장된 상태를 프로그래밍 방식으로 수정 |
| **스레드 독립성** | 서로 다른 `thread_id`는 완전히 독립된 상태 |
| `InMemoryStore` | 스레드 간 공유되는 키-값 장기 메모리 저장소 |
| `compile(store=store)` | 그래프 노드에서 `store` 파라미터로 장기 메모리 접근 |
| `trim_messages` | 토큰 수 기준으로 오래된 메시지를 잘라내어 LLM에 전달 (체크포인트 유지) |
| `RemoveMessage` | 체크포인트에서 특정 메시지를 영구적으로 삭제 |
| **Durable Execution** | 실패 시 마지막 성공 체크포인트에서 재개 |

### 다음 단계
→ **[07_streaming.ipynb](./07_streaming.ipynb)**: 스트리밍을 배웁니다.
